# SV Data Leakage Investigation
## DNA Gene Mapping Project
**Author:** Sharique Mohammad  
**Date:** February 2026  

---

## Objective
Investigate why SV models achieve 100% accuracy:
- Check for target leakage in features
- Analyze feature-target correlations
- Identify problematic features
- Recommend fixes

## Expected Runtime
5-10 minutes (SV data only, small dataset)

---
## 1. Setup

In [ ]:
# Imports
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, pearsonr

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("OK Imports successful")

In [ ]:
# Configuration
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data" / "ml"
FIGURES_DIR = PROJECT_ROOT / "data" / "analytical" / "figures" / "phase3"
REPORTS_DIR = PROJECT_ROOT / "data" / "analytical" / "reports"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("="*80)
print("SV DATA LEAKAGE INVESTIGATION")
print("="*80)

---
## 2. Load SV Datasets

In [ ]:
print("\nLoading SV datasets...")

# Load all SV datasets
with open(DATA_DIR / "sv_train.pkl", 'rb') as f:
    sv_train = pickle.load(f)
    X_train, y_train = sv_train['X'], sv_train['y']

with open(DATA_DIR / "sv_validation.pkl", 'rb') as f:
    sv_val = pickle.load(f)
    X_val, y_val = sv_val['X'], sv_val['y']

with open(DATA_DIR / "sv_test.pkl", 'rb') as f:
    sv_test = pickle.load(f)
    X_test, y_test = sv_test['X'], sv_test['y']

print(f"Train: {X_train.shape}")
print(f"Val: {X_val.shape}")
print(f"Test: {X_test.shape}")
print(f"\nFeatures: {list(X_train.columns)}")

---
## 3. Feature-Target Correlation Analysis

In [ ]:
print("\n" + "="*80)
print("FEATURE-TARGET CORRELATION ANALYSIS")
print("="*80)

# Combine train + val for analysis
X_combined = pd.concat([X_train, X_val], ignore_index=True)
y_combined = pd.concat([pd.Series(y_train), pd.Series(y_val)], ignore_index=True)

# Calculate correlations
correlations = []

for col in X_combined.columns:
    # Pearson correlation
    pearson_r, pearson_p = pearsonr(X_combined[col], y_combined)
    
    # Spearman correlation (rank-based, better for non-linear)
    spearman_r, spearman_p = spearmanr(X_combined[col], y_combined)
    
    correlations.append({
        'feature': col,
        'pearson_r': pearson_r,
        'pearson_p': pearson_p,
        'spearman_r': spearman_r,
        'spearman_p': spearman_p,
        'abs_pearson': abs(pearson_r),
        'abs_spearman': abs(spearman_r)
    })

corr_df = pd.DataFrame(correlations)
corr_df = corr_df.sort_values('abs_pearson', ascending=False)

print("\nFeatures sorted by correlation with target (is_high_risk_sv):")
print(corr_df[['feature', 'pearson_r', 'spearman_r']].to_string(index=False))

# Flag suspicious features
suspicious = corr_df[corr_df['abs_pearson'] > 0.95]

if len(suspicious) > 0:
    print("\n" + "="*80)
    print("SUSPICIOUS FEATURES (correlation > 0.95)")
    print("="*80)
    for _, row in suspicious.iterrows():
        print(f"\n{row['feature']}:")
        print(f"  Pearson r: {row['pearson_r']:.4f}")
        print(f"  Spearman r: {row['spearman_r']:.4f}")
        print(f"  P-value: {row['pearson_p']:.2e}")
        print(f"  WARNING: Likely target leakage!")
else:
    print("\nNo features with correlation > 0.95 found.")

# Save report
corr_df.to_csv(REPORTS_DIR / "sv_feature_correlations.csv", index=False)
print("\nOK Correlations saved: sv_feature_correlations.csv")

---
## 4. Visualize Feature-Target Relationships

In [ ]:
print("\nGenerating correlation heatmap...")

# Create correlation matrix with target
data_with_target = X_combined.copy()
data_with_target['is_high_risk_sv'] = y_combined

corr_matrix = data_with_target.corr()

# Plot heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0, 
            vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('SV Features Correlation Matrix (with Target)', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "27_sv_correlation_heatmap.png", dpi=150, bbox_inches='tight')
plt.close()

print("OK Heatmap saved")

---
## 5. Perfect Prediction Test

In [ ]:
print("\n" + "="*80)
print("PERFECT PREDICTION TEST")
print("="*80)

# Test if any single feature can perfectly predict target
perfect_features = []

for col in X_combined.columns:
    # Try simple threshold-based classification
    unique_vals = sorted(X_combined[col].unique())
    
    if len(unique_vals) <= 10:  # Categorical or low-cardinality
        # Test each unique value as threshold
        for threshold in unique_vals:
            predictions = (X_combined[col] >= threshold).astype(int)
            accuracy = (predictions == y_combined).mean()
            
            if accuracy >= 0.999:  # Near-perfect
                perfect_features.append({
                    'feature': col,
                    'threshold': threshold,
                    'accuracy': accuracy,
                    'rule': f"{col} >= {threshold}"
                })
                break

if perfect_features:
    print("\nFOUND PERFECT PREDICTOR FEATURES:")
    print("="*80)
    for feat in perfect_features:
        print(f"\n{feat['feature']}:")
        print(f"  Rule: {feat['rule']}")
        print(f"  Accuracy: {feat['accuracy']:.6f}")
        print(f"  CONCLUSION: This feature IS the target (or derived from it)")
else:
    print("\nNo single feature can perfectly predict the target.")
    print("Perfect accuracy may come from feature combinations.")

---
## 6. Feature Value Distribution by Target

In [ ]:
print("\nAnalyzing feature distributions by target class...")

# Plot top 5 most correlated features
top_features = corr_df.head(5)['feature'].tolist()

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, feat in enumerate(top_features):
    ax = axes[idx]
    
    # Plot distributions
    low_risk = X_combined[y_combined == False][feat]
    high_risk = X_combined[y_combined == True][feat]
    
    ax.hist(low_risk, bins=30, alpha=0.5, label='Low-risk', density=True)
    ax.hist(high_risk, bins=30, alpha=0.5, label='High-risk', density=True)
    ax.set_xlabel(feat)
    ax.set_ylabel('Density')
    ax.legend()
    ax.grid(True, alpha=0.3)

# Remove extra subplot
fig.delaxes(axes[5])

plt.suptitle('Top 5 Features - Distribution by Target Class', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "28_sv_feature_distributions.png", dpi=150, bbox_inches='tight')
plt.close()

print("OK Distribution plots saved")

---
## 7. Leakage Investigation Report

In [ ]:
print("\n" + "="*80)
print("LEAKAGE INVESTIGATION REPORT")
print("="*80)

report = []
report.append("SV DATA LEAKAGE INVESTIGATION REPORT")
report.append("="*80)
report.append("")
report.append("SUMMARY:")
report.append(f"  Total features: {len(X_combined.columns)}")
report.append(f"  Highly correlated (>0.95): {len(suspicious)}")
report.append(f"  Perfect predictors: {len(perfect_features)}")
report.append("")

if len(suspicious) > 0 or len(perfect_features) > 0:
    report.append("CONCLUSION: DATA LEAKAGE DETECTED")
    report.append("")
    report.append("PROBLEMATIC FEATURES:")
    
    for _, row in suspicious.iterrows():
        report.append(f"  - {row['feature']} (r={row['pearson_r']:.4f})")
    
    for feat in perfect_features:
        report.append(f"  - {feat['feature']} (perfect predictor)")
    
    report.append("")
    report.append("RECOMMENDATIONS:")
    report.append("  1. Review feature engineering in Databricks")
    report.append("  2. Check if these features are derived from target")
    report.append("  3. Remove leaked features and retrain models")
    report.append("")
    report.append("LIKELY CAUSES:")
    report.append("  - Features calculated using target variable")
    report.append("  - Features that encode target in different form")
    report.append("  - Score/risk features derived from target labels")
else:
    report.append("CONCLUSION: NO OBVIOUS LEAKAGE DETECTED")
    report.append("")
    report.append("Perfect accuracy may be due to:")
    report.append("  1. Feature combinations (not single features)")
    report.append("  2. Very simple classification task")
    report.append("  3. Small dataset with clear patterns")

report.append("")
report.append("="*80)

# Print and save report
report_text = "\n".join(report)
print("\n" + report_text)

report_file = REPORTS_DIR / "sv_leakage_investigation.txt"
with open(report_file, 'w') as f:
    f.write(report_text)

print(f"\nOK Report saved: {report_file.name}")

---
## 8. Summary

In [ ]:
print("\n" + "="*80)
print("FILES CREATED")
print("="*80)
print("\nReports:")
print("  - sv_feature_correlations.csv")
print("  - sv_leakage_investigation.txt")
print("\nFigures:")
print("  - 27_sv_correlation_heatmap.png")
print("  - 28_sv_feature_distributions.png")

print("\n" + "="*80)
print("SV LEAKAGE INVESTIGATION COMPLETE")
print("="*80)

if len(suspicious) > 0 or len(perfect_features) > 0:
    print("\nACTION REQUIRED:")
    print("  1. Review flagged features in Databricks feature engineering")
    print("  2. Remove leaked features from ml_dataset tables")
    print("  3. Re-run Phase 3 with clean features")
else:
    print("\nNo obvious leakage found.")
    print("Perfect accuracy likely due to feature combinations.")
    print("SV task may be genuinely easy to solve.")

print("="*80)